# 00 — Documentação do projeto
## Terra Signal: previsão e prevenção de churn
**Hackathon Databricks Academia · ASN.Rocks · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

### Histórico do documento

| Versão | Data | Responsável | Descrição |
|---|---|---|---|
| 1.0 | 09/11/2026 | André Ferreira | Versão inicial: problema, dados, preparação e análise exploratória |
| 1.1 | 10/11/2026 | André Ferreira | Respostas dos organizadores, decisões do grupo (checklist D1 a D19), ABT na camada gold e seleção de variáveis sugerida (notebooks 04 e 05) |
| 1.2 | 11/11/2026 | André Ferreira | Notebooks reorganizados (05 a 08) com o modelo do Antonio; execução oficial: logística Lasso, segmentação e arquivo da entrega (D19 a D23) |
| 1.3 | 12/11/2026 | André Ferreira | Monitoramento do modelo (09) e motor de atendimento (10), com o app Central de retenção no Databricks Apps (D24 e D25); registro do modelo no experimento fixo `/Shared/terra_signal_churn` |

**Siglas:** IV (*Information Value*), PSI (*Population Stability Index*), ABT (*Analytical Base Table*), AUC (*Area Under
the Curve*) e as demais estão por extenso no glossário do início dos notebooks 03, 04 e 05, com o lugar onde cada uma é
explicada.

## Sumário executivo

A Terra Signal está perdendo clientes com o aumento da concorrência e não tem nenhuma previsibilidade sobre o
cancelamento. Este projeto constrói uma solução de ponta a ponta para identificar, na carteira, os clientes com maior
risco de sair, para que o time de vendas ofereça planos de retenção durante as ligações.

**O que já sabemos:**
- **26,7% dos clientes cancelaram**, e eles respondiam por **30,8% da receita mensal**: quem sai paga mais.
- O risco se concentra em **contrato mensal, poucos meses de casa, fibra ótica, falta de suporte técnico ou segurança e
  pagamento por cheque eletrônico**. São pontos que a empresa pode mudar com uma oferta.
- O comentário do cliente foi escrito **depois** do cancelamento e não pode ser usado para prever (vazamento).

**O que está pronto (11/11):**
- Dados limpos e documentados, com as decisões do grupo registradas num checklist (D1 a D25) que abre cada notebook.
- A **ABT** (*Analytical Base Table*, a tabela que o modelo lê) gravada na camada gold, com um dicionário que sugere o
  uso de cada coluna.
- O **modelo final**: regressão logística Lasso sobre 41 colunas, escolhida entre seis modelos pela consistência
  (notebook 06). Em clientes que ele não viu: **ROC-AUC de 0,830**, recall de 78% e precisão de 52%.
- **Para a retenção:** os 20% de maior risco concentram **48% de quem sai** (2,4 vezes melhor que ao acaso).
- **Na carteira a pontuar:** 571 de 1.409 clientes com sinal de saída; 398 deles são de mensalidade alta ("salvar já"),
  369 no segmento de fibra no contrato mensal com pouca proteção (notebook 07).
- O **arquivo da entrega**, `prediction_grupo4_<data_hora>.csv`, conferido nas 8 regras (notebook 08).
- Uma **conferência da seleção** com os métodos da aula (11 variáveis) que, numa regressão logística, chega a **AUC de 0,841**
  (*Area Under the Curve*: a chance de o modelo dar nota de risco maior a quem sai do que a quem fica), contra 0,678 de
  uma regra simples sem modelo.

## Introdução: os 7 passos e onde cada um está

| Passo | Onde está | Situação |
|---|---|---|
| 1. Definição do problema de negócio | Este documento, seção 1 | Feito |
| 2. Aquisição dos dados | Este documento, seção 2, e notebook 01 — Dicionário de dados | Feito |
| 3. Preparação dos dados | Notebook 02 — Preparação dos dados | Feito |
| 4. Análise exploratória | Notebook 03 — Análise exploratória | Feito |
| 5. Modelagem | Notebooks 04 — Variáveis derivadas e ABT, 05 — Seleção de variáveis, 06 — Modelagem e 07 — Segmentação | Feito |
| 6. Visualização e comunicação | Notebooks 07 (segmentação) e 08 (números da apresentação); app Central de retenção (atendimento, campanha, fila e monitoramento); apresentação de 15 minutos | Feito (falta ensaiar) |
| 7. Deploy e manutenção | Pipeline no MLflow (06), arquivo da entrega (08), monitoramento (09), motor de atendimento (10) e app no Databricks Apps; seção 7 | Feito (falta comparar o arquivo com o notebook oficial) |

**Arquitetura medalhão no Unity Catalog** (`workspace.terra_signal`):

| Camada | Tabelas | Notebook |
|---|---|---|
| Bronze (dado bruto) | `bronze_history`, `bronze_inference` | 01 |
| Silver (dado limpo) | `silver_history`, `silver_inference` | 02 |
| Dicionário | `dicionario_dados`, `dicionario_categorias` | 03 |
| Gold (pronto para o modelo) | `gold_churn_treino`, `gold_churn_inferencia`, `gold_churn_dicionario`, `gold_churn_texto` | 04 |
| Gold (seleção de variáveis) | `gold_churn_split`, `gold_churn_variaveis_modelo`, `gold_churn_selecao` | 05 |
| Gold (previsões) | `gold_churn_abt_scored`, `gold_churn_inferencia_scored`, `gold_churn_modelo_info` | 06 |
| Gold (segmentos) | `gold_churn_segmentos`, `gold_churn_segmentos_perfil` | 07 |
| Gold (monitoramento) | `gold_churn_monitoramento`, `gold_churn_monitoramento_distribuicao`, `gold_churn_fila_retencao` | 09 |
| Gold (atendimento) | `gold_churn_ofertas`; e, na pasta do app, `modelo_motor.json`, `clientes_atendimento.csv`, `ganho_teste.csv` e `relatos_inferencia.csv` | 10 |
| Leitura dos comentários (LLM) | `feedback_entrada_v2`, `feedback_textos_v2`, `feedback_clientes_v2` (em `workspace.default`) | 11 |

## 1. Definição do problema de negócio

| Pergunta | Resposta |
|---|---|
| Quem é o cliente? | Terra Signal (TS), telecom fundada em 2017: telefonia móvel, fibra ótica e IoT |
| Qual é a dor? | Mais cancelamentos com o aumento da concorrência e nenhuma previsibilidade; a empresa não sabe para onde direcionar os recursos |
| O que a TS precisa? | Identificar os clientes da carteira com risco de sair e, se possível, dar acesso ao time de vendas para oferecer retenção nas ligações |
| O que é o Y? | `Churn`: 1 = cancelou, 0 = ficou |
| Que decisão o modelo apoia? | Quem acionar para retenção, e com que oferta |
| O que é mais importante? | Segundo a TS, que o modelo funcione e seja acessível de alguma forma |

Perguntas ainda sem resposta: ver "Perguntas para os organizadores" na seção 5.

## 2. Mapeamento das fontes de dados

| Arquivo | Clientes | Colunas | Papel |
|---|---|---|---|
| `history.csv` | 5.634 | 23 | Histórico com o Y (`Churn`): base de treino |
| `inference.csv` | 1.409 | 22 | Clientes a pontuar, sem o Y: base do `Prediction.csv` |

O slide do case fala em 5.912 e 1.461 linhas: a diferença são comentários do feedback que ocupam mais de uma linha
no arquivo. Os clientes são 5.634 e 1.409.

| Grupo | Variáveis |
|---|---|
| Dados do cliente | `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `MonthlyIncome` |
| Relacionamento | `tenure` (meses de casa), `Contract`, `PaperlessBilling`, `PaymentMethod` |
| Serviços | `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` |
| Financeiro | `MonthlyCharges`, `TotalCharges` |
| Texto | `CustomerFeedback` (comentário do cliente, em inglês) |
| Identificação e alvo | `customerID`, `Churn` |

**Moeda:** real (R$), em todo o dataset (confirmado pelos organizadores em 10/11).

O dicionário completo (o que cada coluna é, tipo, domínio esperado, o que foi verificado nos dados e o que foi
confirmado pelos organizadores) está no notebook 01, seção 3. O notebook 03 grava a versão final no catálogo (`workspace.terra_signal.dicionario_dados`).

## 3. Preparação dos dados (notebooks 01 e 02)

O notebook 01 encontrou os problemas; o grupo decidiu o tratamento em 10/11 (checklist D1 a D8 e D11); o notebook 02
aplicou. Só três colunas tinham problema.

| Problema | Casos (histórico + inferência) | Decisão | Código |
|---|---|---|---|
| `PhoneService` em minúsculo | 70 + 23 | Padronizado para `Yes`/`No` (só erro de grafia, segundo os organizadores) | D1 |
| `TotalCharges` com `null` | 278 + 64 | Preenchido com `MonthlyCharges` × `tenure`, como aproximação; 0 quando `tenure` = 0 | D2, D3 |
| `tenure` com `unknown` | 19 + 4 | Recuperado por `TotalCharges` ÷ `MonthlyCharges`; mediana do histórico no único caso sem os dois | D4 |
| Ausência de `TotalCharges` | 278 | Indicador `fl_total_charges_ausente` (quem não tinha o valor saiu mais: 33,1% contra 26,4%) | D5 |
| Ausência de `tenure` | 19 | Sem indicador: poucos casos, sem diferença significativa | D6 |
| `TotalCharges` fora de ±20% da regra | 47 + 9 | Mantidos (mudança de plano e promoções, segundo os organizadores) | D7 |
| Quebras de linha no comentário | 85 + 15 | Trocadas por espaço | D8 |

A fonte única do grupo são as tabelas bronze (D11). As tabelas limpas ficam em `silver_history` e `silver_inference`.

## 4. Análise exploratória (notebook 03)

| Pergunta | O que encontramos |
|---|---|
| Quanto churn existe? | 26,7% dos clientes, com 30,8% da receita mensal |
| Quem sai? | Contrato mensal (42,7% saem, contra 3,3% no de dois anos), cheque eletrônico (45,1%), fibra ótica (42,1%), sem suporte técnico ou segurança (cerca de 42%) |
| Os valores mudam? | Quem sai é novo de casa (mediana de 10 meses, contra 37) e paga mais (R$ 80, contra R$ 64). A renda é igual |
| O que mais separa? | Contrato (IV 1,17), meses de casa (0,77) e serviços de proteção (cerca de 0,7) |
| O que não ajuda? | Gênero, telefonia, linhas múltiplas e renda (IV abaixo de 0,02) |
| O que se repete? | `MonthlyCharges` com o pacote (0,91 com o tipo de internet); `TotalCharges` com meses de casa (0,89); `PhoneService` com `MultipleLines` (1,00); os serviços só parecem ligados por causa de "No internet service" |
| A inferência é parecida com o histórico? | Sim: nenhuma variável mudou (PSI máximo de 0,011); churn esperado de cerca de 27% |
| Onde o risco se concentra? | Cliente novo no contrato mensal: 55% saem nos primeiros 6 meses; no contrato de dois anos, quase ninguém sai |
| O que fica fora? | O comentário: 25% dos textos contam que o cliente já cancelou, e uma regra de uma linha acerta 98,9% |

**O que muda na leitura depois do notebook 04:** o poder dos streamings vinha só da categoria "No internet service".
Entre quem tem internet, assinar ou não streaming não muda o risco (33% contra 31% de churn).

## 4b. ABT e seleção de variáveis (notebooks 04 e 05)

**A ABT na camada gold (D14):** uma linha por cliente, todas as variáveis candidatas, sem valores ausentes e com as
mesmas colunas no treino e na inferência. O texto do comentário fica de fora, numa tabela separada (D9).

| Tabela | Conteúdo |
|---|---|
| `gold_churn_treino` | 5.634 clientes, 49 variáveis candidatas e `Churn` (1 = saiu) |
| `gold_churn_inferencia` | 1.409 clientes, as mesmas colunas, sem `Churn` |
| `gold_churn_dicionario` | Uma linha por coluna: origem, fórmula, IV e a sugestão (9 recomendadas, 21 candidatas, 19 que sugerimos descartar) |
| `gold_churn_texto` | O comentário e o indicador de cancelamento, só para o modelo com texto (contraexemplo) |
| `gold_churn_split` e `gold_churn_variaveis_modelo` | Quem é treino e quem é teste, e as 41 colunas do modelo (05) |
| `gold_churn_selecao` | A conferência com os métodos da aula (11 variáveis) |

**Seleção de variáveis (D19, notebook 05):** a seleção final usa testes estatísticos com correção para múltiplos
testes e retira as duplicatas exatas: ficam **41 colunas** (seções 2 e 3). A conferência com os métodos da aula
(seções 4 e 5) parte das 9 recomendadas e testa cada candidata numa regressão logística, com validação cruzada
repetida; entra a que mais melhora a AUC, até o ganho ficar abaixo de 0,002:

| Modelo | AUC |
|---|---|
| Regra sem modelo: contrato mensal e cheque eletrônico | 0,678 |
| As 9 recomendadas | 0,834 |
| **Conferência: as 9 + cliente novo no contrato mensal + cheque eletrônico** | **0,841** |
| Todas as 30 recomendadas e candidatas | 0,843 |
| Seleção + texto do comentário (vazamento) | 0,993 |

As variáveis extras quase não acrescentam, e a repetição entre as selecionadas é aceitável (VIF, *Variance Inflation
Factor*, máximo de 2,75). As 11 variáveis da conferência estão todas dentro das 41 colunas do modelo (seção 6 do 05).

## 5. Modelagem (notebooks 05 a 08)

| Item | O que fizemos |
|---|---|
| Base | `gold_churn_treino` (notebook 04), com 20% dos clientes guardados para o teste e as 41 colunas do notebook 05 |
| Modelos | Seis modelos comparados (logística Lasso, Ridge e Elastic Net, *random forest*, XGBoost e CatBoost); escolhida a logística Lasso (C = 0,3), a mais consistente entre treino, teste e base total; a Ridge ficou a 0,003. A apresentação usa a Ridge (C = 0,03, execução do Antonio): na mesma separação, as duas dão a mesma fila (222 dos 225 clientes do topo em comum; `DOC_MODELO.md`, seção 4b) |
| Texto do feedback | Modelo oficial **sem** o texto; o com texto vira contraexemplo na apresentação |
| Métricas | AUC para comparar modelos; precisão e recall no limiar escolhido; comparar com regras simples (por exemplo, "contrato mensal") |
| Resultado no teste | ROC-AUC 0,830, KS 0,523, recall 78%, precisão 52%, F1 0,624; limiar de 0,5046 (máximo do F1 fora da dobra) |
| Saída | Probabilidade de churn por cliente (`gold_churn_inferencia_scored`, notebook 06) e o `prediction_grupo4_<timestamp>.csv` com `customerID` e `prediction` (Yes/No) para os 1.409 clientes (notebook 08) |

### Como avaliar e escolher o modelo

**A dúvida:** o modelo de propensão devolve uma probabilidade. Como dizer se ele acertou? Basta a AUC, com a base
desbalanceada? E, para o limiar, o que importa mais: alcançar mais clientes que vão sair (gastando mais com contato) ou
errar menos e acionar menos gente?

**Teoria aplicada: cada métrica responde a uma pergunta diferente.**

| Pergunta | Métrica | Depende do limiar? | Uso aqui |
|---|---|---|---|
| O modelo **ordena** bem quem sai na frente de quem fica? | AUC-ROC: a chance de um cliente que saiu receber probabilidade maior que um que ficou | Não | Comparar algoritmos e evitar sobreajuste (treino × validação) |
| Ele ordena bem **olhando só os que saem**? | AUC-PR (precisão × recall), cuja linha de base é a taxa de churn (0,27) | Não | Complementa a AUC quando a classe positiva é minoria |
| A **probabilidade** está certa? Se o modelo diz 30%, saem 30%? | Brier e curva de calibração | Não | Obrigatória se entregarmos a propensão |
| Quanto melhor que o sorteio no **topo da lista**? | Lift e ganho acumulado por decil | Não | Mensagem para o negócio: "nos 10% de maior risco, saem X vezes mais que a média" |
| Quantos acerto **no limiar escolhido**? | Precisão, recall, F1 e matriz de confusão | Sim | Para o arquivo Yes/No da entrega |

**Sobre o desbalanceamento:** 26,7% de churn é um desbalanceamento moderado. A AUC-ROC não muda com a proporção das
classes, por isso serve para comparar modelos, mas sozinha não diz quantos clientes acionar nem quanto se acerta entre
eles. A acurácia é a métrica que engana: chutar "ficou" para todos dá 73,3% (7.3b).

**Sobre o limiar (alcançar mais ou errar menos):** a escolha não é estatística, é de custo. Ligar para um cliente
compensa quando

> **acionar se p × s × V > c, ou seja, se p > p\* = c ÷ (s × V)**

em que *p* é a probabilidade de churn prevista, *s* a parte dos clientes em risco que a oferta retém, *V* o valor do
cliente (R$ 896 por ano, seção 10.1 do 03) e *c* o custo de acionar. A tabela abaixo mostra o p* em cenários:

| Custo por cliente acionado | Sucesso 10% | Sucesso 20% | Sucesso 40% |
|---|---|---|---|
| R$ 2 (e-mail ou app) | 0,022 | 0,011 | 0,006 |
| R$ 10 (ligação) | 0,112 | 0,056 | 0,028 |
| R$ 50 (ligação + brinde) | 0,558 | 0,279 | 0,140 |
| R$ 150 (desconto) | acima de 1: nunca compensa | 0,837 | 0,419 |


- **Contato barato** (e-mail ou aviso no app, R$ 2): o limiar fica abaixo de 3%. Compensa acionar quase todos: **priorizar recall**.
- **Ligação** (R$ 10) com 20% de sucesso: limiar perto de 6%, ainda muito baixo: recall continua prioritário.
- **Oferta cara** (desconto de R$ 150) com 20% de sucesso: limiar de 84%. Aí só os de maior risco: **priorizar precisão**.
- **Capacidade limitada** (o time só consegue ligar para N clientes por mês): o limiar vira "os N de maior probabilidade",
  e a métrica é a precisão e o recall nesse top N. 

**Proposta para o grupo:**
1. **Escolher o algoritmo** pela AUC-ROC e pela AUC-PR na validação cruzada, conferindo a diferença entre treino e teste.
2. **Conferir a calibração** (Brier e curva) do escolhido, porque a propensão é o produto que vamos mostrar.
3. **Escolher o limiar** pela regra de custo, se os organizadores derem o custo e a taxa de sucesso; se não, pelo
   limiar que maximiza o F1 na validação, informando a precisão e o recall que ele dá. Uma checagem simples: a
   proporção de "Yes" na inferência ficou em 40,5%, acima dos 27% de churn real, por causa do balanceamento das classes
   e do limiar no máximo do F1 (recall de 78%); treino, teste e inferência têm a mesma taxa prevista, sem sinal de mudança.
4. **Entregar as duas coisas:** a probabilidade para o negócio ordenar a carteira e o Yes/No que o arquivo da entrega pede.
5. **Comparar o modelo com a linha de base (seção 10.2 do 03):** a curva de ganho dele precisa passar acima das regras simples.


### Perguntas para os organizadores

Respostas recebidas do Carlos (Databricks) em 10/11 estão marcadas; as demais continuam em aberto.

**Entrega e avaliação**
1. Como o arquivo de previsão é avaliado: qual métrica (acurácia, F1, AUC, recall) e sobre o Yes/No ou sobre a probabilidade?
2. O arquivo deve ter os 1.409 clientes da inferência (e não as 1.461 linhas físicas)? Colunas `customerID` e `prediction` com Yes/No?
3. A avaliação considera o vazamento do `CustomerFeedback`? Um modelo que usa o texto é penalizado, aceito ou esperado?

**Negócio**
4. Que ação de retenção o time de vendas faz: ligação, e-mail, oferta no app, desconto? Quanto custa por cliente?
5. Quantos clientes o time consegue acionar por mês (capacidade)?
6. Existe uma taxa de sucesso esperada para a retenção? Sem ela, usamos cenários (ver acima).
7. O objetivo é reduzir a quantidade de cancelamentos ou a receita perdida? Clientes de mensalidade alta pesam mais?

**Dados**
8. O que é um cliente ativo, e em que janela o churn foi medido (por exemplo, cancelou nos 30 dias seguintes)?
9. Como foram separados histórico e inferência: sorteio ou por data?
10. ~~`SeniorCitizen`: qual idade define idoso? `MonthlyIncome` e `MonthlyCharges`: qual moeda?~~ **Respondida (10/11):** idoso = 65 anos ou mais; moeda R$ em todo o dataset; `MonthlyIncome` é renda familiar, de base externa; 72 meses é teto real; `PhoneService` minúsculo é só erro de grafia.
11. O `CustomerFeedback` foi coletado antes ou depois do cancelamento?

**Apresentação**
12. Quem é a banca e o que ela espera ver: resultado de negócio, detalhe técnico ou os dois? Quanto tempo temos?
13. Podemos levar um material técnico separado (um HTML de documentação) para perguntas da banca?

## Decisões do grupo

Todas as decisões estão no **checklist de decisões (D1 a D25)**, a mesma tabela no início de cada notebook, com as
opções, a proposta, o impacto, a evidência e o status:
- **☑ decidido** (com a data): preparação dos dados (D1 a D8), fonte única (D11), texto fora do modelo oficial (D9),
  linhas da entrega (D13), formato da gold (D14), fonte do 03 (D17) e divisão de papéis (D18);
- **☑ decididos na modelagem (11/11):** seleção de variáveis (D19, 41 colunas), modelo final (D21, logística Lasso),
  limiar (D22, 0,5046) e arquivo da entrega (D23);
- **◐ sugestão:** formato dos serviços (D10), descartes (D12 e D15) e variáveis de preço (D16), resolvidos pela seleção
  do 05 e pela regularização do 06; e a segmentação (D20), para o negócio;
- **◐ proposta (12/11):** monitoramento do modelo (D24, notebook 09) e motor de atendimento (D25, notebook 10), os dois
  no app Central de retenção; e a leitura do comentário por LLM (D26, notebook 11), no app Central de relacionamento.

O resumo de cada decisão, com quem respondeu e o motivo, está no arquivo de decisões do projeto.

## 6. Visualização e comunicação

- **Apresentação de 15 minutos** para a banca: Carlos Dip e Victor Rodrigues (Databricks), Adriana Silva (ASN.Rocks) e
  Daniele Nobrega (Credsystem).
- **Linha da apresentação:** o problema e quanto ele custa; quem sai e por quê; o modelo e quanto ele acerta a mais que
  regras simples; como o time de vendas vai usar, mostrado ao vivo no app.
- **App Central de retenção** (Databricks Apps, pasta `app_monitoramento`), com oito páginas, legenda de cores e o botão "Como interpretar" em cada gráfico e tabela:

| Página | Para quem | O que mostra |
|---|---|---|
| Atendimento | Operadora da central | Receptivo (o cliente ligou) e ativo (a lista de ligações da carteira, separada em retenção, recuperação e fidelização pelo último relato e ordenada pela receptividade e pela nota): risco, motivos, relato, histórico de contatos, escada de ofertas e simulador; para quem ainda não é cliente, o pacote de menor risco. Registro com roteiro da ligação, missões do dia, XP e conquistas |
| Plano de marketing | Marketing | O catálogo de ofertas (ativa, fala, validade, segmentos), que a escada do atendimento segue na hora, e quantos clientes da fila o plano tira da zona de risco |
| Campanha de retenção | Gestor de retenção | Quantos clientes contatar para o maior lucro, com custo, aceite e desconto informados; conta decil a decil |
| Fila de retenção | Gestor de retenção | Matriz risco × valor e a fila do lote, com segmento e oferta |
| Saúde do modelo | Time de dados | Semáforo de cada lote e as ações necessárias (acompanhar, investigar, retreinar) |
| Deriva das variáveis | Time de dados | PSI de cada variável e a mudança faixa a faixa, com o lote simulado |
| Ficha do modelo | Todos | Modelo, limiar, métricas, curva de ganho e regras de retreino |
| Pergunte aos dados | Gestores | Perguntas em português respondidas pelo Genie, em formato de conversa, com tabela, gráfico e SQL |

- O material técnico (estes notebooks e o `DOC_MODELO.md`) fica disponível para perguntas.

## 7. Deploy e manutenção

A TS citou três formas de acesso (sugestões): uma API de inferência em tempo real (Databricks Model Serving), um chatbot
para consultar os clientes (Databricks Genie) e um app web para calcular o risco (Databricks Apps).

| Forma de acesso | Situação | Onde |
|---|---|---|
| App web (Databricks Apps) | **Feito:** app Central de retenção, com atendimento, campanha, fila e monitoramento | Pasta `app_monitoramento`; notebooks 09 e 10 |
| App web (Databricks Apps, FastAPI) | **Feito:** app Central de relacionamento (Lucas), com a carteira por risco, o perfil do cliente numa tela e a leitura do comentário pela LLM | Pasta `app_central_relacionamento`; notebook 11 |
| Chatbot (Databricks Genie) | **Feito:** espaço *Terra Signal · Retenção* sobre 6 tabelas gold, com instruções e perguntas em português; também dentro do app (página Pergunte aos dados) | Tabelas de 06, 07, 09 e 10 |
| API em tempo real (Model Serving) | Proposta: o pipeline já está no MLflow e pode virar um *endpoint* | Notebook 06, seção 6 |

**O que já está feito:**
- **Modelo em produção:** o pipeline (limpeza + modelo) está registrado no MLflow, no experimento fixo
  `/Shared/terra_signal_churn`, e validado: recebe a base no formato da ABT e reproduz as previsões (06).
- **Previsões no catálogo:** a probabilidade, a previsão e o decil de cada cliente (`gold_churn_inferencia_scored`), com o
  segmento e a ação sugerida (`gold_churn_segmentos`, 07).
- **Monitoramento (09, D24):** PSI das variáveis e das notas, taxa de Yes, reprodução pelo MLflow e qualidade dos dados,
  com semáforo; um lote simulado prova que o alarme acende quando a carteira muda; cada indicador aceso vira uma ação no app.
- **Motor de atendimento (10, D25):** a mesma conta do modelo oficial, exportada para o app e conferida (diferença de nota
  zero); a escada de ofertas de cada cliente; a tabela de preço recuperada dos dados (99,9% da mensalidade); e a taxa
  real de churn por decil, que alimenta a curva de ganho e o simulador de campanha.

**Manutenção:** a cada nova carteira, rodar o 06 (pontuação), o 09 e o 10. As regras de retreino estão na seção 6 do 09:
variável do núcleo em alerta, taxa de Yes a mais de 10 pontos do teste ou recall real 10 pontos abaixo de 78%.

**Próximo passo (histórico de contatos e uplift):** o registro dos atendimentos no app forma o histórico de contatos de cada cliente (quantas ligações, com que motivo, com que resultado) e mede o aceite e a permanência de quem recebeu cada oferta. O catálogo do marketing vira uma tabela no Unity Catalog.
Com esses dados, dá para passar da previsão de churn ao modelo de *uplift*: qual oferta funciona para qual cliente.

## Entrega

| Item | Como |
|---|---|
| Parte técnica | Repositório no GitHub (fork do repo do hackathon), organizado, com o nome da equipe e os integrantes no README |
| Previsão | `prediction_grupo4_<timestamp>.csv` (notebook 08), na raiz do repositório; o link do repositório vai no Google Forms de submissão |
| Prazo | Commits feitos depois do horário de finalização não são considerados |
| Apresentação | 15 minutos, no segundo dia, presencial (pelo menos um integrante) |

## Conclusões

Os dados são bons e o perfil de quem sai é claro: o churn está ligado a **como o cliente se relaciona com a empresa**
(contrato, tempo de casa, serviços e forma de pagamento), e não a quem ele é. Isso é uma boa notícia para o negócio,
porque são pontos que uma oferta de retenção pode mudar. A ABT está na camada gold, conferida; 41 colunas entram no modelo, e os métodos
da aula confirmam as principais. O modelo final é uma regressão logística Lasso, escolhida entre seis pela
consistência, com o limiar que maximiza o F1 (notebook 06): ROC-AUC de 0,830 no teste, e os 20% de maior risco
concentram 48% de quem sai. Os segmentos (07) dizem que oferta faz sentido para cada
grupo em risco, e o arquivo da entrega sai do 08, respeitando a linha do tempo: só informação disponível antes do
cancelamento. O app Central de retenção leva tudo isso para quem usa: a operadora vê o risco, o porquê e a oferta de cada cliente na
hora da ligação, o gestor dimensiona a campanha pelo lucro e o time de dados acompanha a saúde do modelo (09 e 10).